# DFT of the hBN trilayer test: monolayer, AA' substrate, AA'A trilayer

**Goal.** Second benchmark of the effective scheme: the top layer of an AA'A trilayer, with the two layers below as
the substrate. Three systems, all in the same cell:
- `ML_top`: the monolayer A alone, in the position of the top layer (layer of the effective scheme);
- `AA1`: the AA' bilayer, in the position of the two lower layers (substrate: only its screening is needed);
- `AA1A`: the AA'A trilayer (explicit reference). A' is the layer with B and N exchanged (rotated by 180° about z);
  the top layer A sits above the bottom one, B over B and N over N, as in bulk hBN.

Layers at $z = -d/2$ (A), $+d/2$ (A'), $+3d/2$ (A), with $d$ = 3.33 Å.

**Taken from `hBN_bilayer`** (not checked again): SG15 ONCV PBE pseudopotentials, no SOC; $a$ = 2.504 Å and $d$ = 3.33 Å
fixed; ecut 80 Ry, scf on a 12x12x1 grid; nscf for Yambo on the 18x18x1 grid with `cg` diagonalization and
`force_symmorphic`. **New**: cell height $c$ = 50 bohr instead of 40, because the trilayer does not fit in half of the
40 bohr cell, which the slab cutoff of Yambo requires (check below, on the density of the trilayer).

The other layers of the trilayer, and the screening of the layers in other positions, are obtained by translations and
rotations (of the KS states with `WfcParser.transform`, of the screening with `YamboScreeningParser.transform`),
validated in `hBN_bilayer`.

In [ ]:
from mppi import InputFiles as I, Calculators as C, Datasets as D, Parsers as P
from mppi.Datasets import PostProcessing as PP
from mppi.Utilities import Constants as Const
import matplotlib.pyplot as plt
import numpy as np
import os

module = '/home/dalessandro/module_script/qe_module'
def qe_code(nodes=1, partition='all12h', time='11:59:00'):
    rr = C.RunRules(scheduler='slurm', partition=partition, memory='125000', time=time, nodes=nodes,
                    ntasks_per_node=16, cpus_per_task=2, omp_num_threads=2, pre_processing=module)
    return C.QeCalculator(rr, activate_BeeOND=True)

In [ ]:
run_dir = 'DFT_analysis'
pseudo_dir = '../../pseudos'                 # relative to run_dir, where pw.x runs
Bohr = 1. / Const.Bohr_to_Angstrom           # bohr per angstrom
alat_ang, d_ang = 2.504, 3.33
alat = alat_ang * Bohr                       # bohr
ecut, k_scf, c = 80, 12, 50.0                # Ry, scf grid, cell height (bohr)

SA, SB = np.array([0., 1. / np.sqrt(3.)]), np.array([1. / 2., 1. / (2 * np.sqrt(3.))])   # sites, units of alat
def layer(kind, z_ang):
    # A: B on SA, N on SB; A': B and N exchanged
    z = z_ang / alat_ang
    sites = [('B', SA), ('N', SB)] if kind == 'A' else [('N', SA), ('B', SB)]
    return [[name, np.array([s[0], s[1], z])] for name, s in sites]

layers = {'bot': layer('A', -d_ang / 2), 'mid': layer("A'", d_ang / 2), 'top': layer('A', 3 * d_ang / 2)}
composition = {'ML_top': ['top'], 'AA1': ['bot', 'mid'], 'AA1A': ['bot', 'mid', 'top']}
systems = list(composition)
nocc = {s: 4 * len(composition[s]) for s in systems}          # occupied bands (8 valence electrons per layer)

def build_input(system, kpoints, nbnd=None):
    inp = I.PwInput()
    inp.set_scf(force_symmorphic=True, conv_thr=1e-8)
    inp.set_num_bnds(nbnd if nbnd is not None else 2 * nocc[system])
    inp.set_pseudo_dir(pseudo_dir=pseudo_dir)
    inp.add_atom(atom='B', pseudo_name='B_ONCV_PBE-1.2.upf', mass=10.811)
    inp.add_atom(atom='N', pseudo_name='N_ONCV_PBE-1.2.upf', mass=14.007)
    pos = [a for l in composition[system] for a in layers[l]]
    inp.set_atoms_number(len(pos))
    inp.set_atomic_positions(pos, type='alat')
    inp.set_lattice(ibrav=4, celldm1=alat, celldm3=c / alat)
    inp.set_energy_cutoff(ecut)
    inp.set_kpoints(type='automatic', points=[kpoints, kpoints, 1])
    return inp

for s in systems:
    print('%-6s' % s, '; '.join('%s (%.3f, %.3f, %+.3f A)' % (n, *(p[:2] * alat_ang), p[2] * alat_ang)
                                for n, p in [a for l in composition[s] for a in layers[l]]))

## Ground states

In [ ]:
study = D.Dataset(run_dir=run_dir, num_tasks=2)
study.set_postprocessing_function(PP.pw_parse_data)
for s in systems:
    inp = build_input(s, k_scf)
    inp.set_prefix('gs_converged-%s' % s)
    study.append_run(id=s, input=inp, runner=qe_code(), name='gs_converged-%s' % s)
res = study.run()
gs = {s: res[i] for i, s in enumerate(systems)}

In [ ]:
# gaps, compared with the c = 40 bohr runs of hBN_bilayer (same layers: ML_bot = layer A, AA1 = the same bilayer)
ref = {'ML_top': 'ML_bot', 'AA1': 'AA1'}
def gap_gamma(r, s):
    ev = np.array(r.get_evals(verbose=False))[0]          # first k point of the grid = Gamma
    return ev[nocc[s]] - ev[nocc[s] - 1]
for s in systems:
    g = gs[s].get_gap(verbose=False)
    line = '%-6s gap %.4f | direct %.4f | at Gamma %.4f eV' % (s, g['gap'], g['direct_gap'], gap_gamma(gs[s], s))
    if s in ref:
        r40 = P.PwParser(os.path.join('..', 'hBN_bilayer', 'DFT_analysis', 'gs_converged-%s.xml' % ref[s]), verbose=False)
        g40 = r40.get_gap(verbose=False)
        line += '  (c = 40: %.4f | %.4f | %.4f)' % (g40['gap'], g40['direct_gap'], gap_gamma(r40, s))
    print(line)

## Slab cutoff: density of the trilayer along z

Planar-averaged valence density of the trilayer (`pp.x`, `plot_num = 0`, + `average.x`, one short slurm job, as in
`hBN_bilayer`). Criterion of `hBN_bilayer`: the weight of the pairs of charges at $|z-z'| > L/2$, whose interaction
the cutoff cuts, must be negligible (bilayer at $c$ = 40 bohr: $10^{-10}$). Estimate from superposed densities, before
the run: $3\times10^{-6}$ at $c$ = 40 bohr, $5\times10^{-10}$ at $c$ = 50 bohr.

In [ ]:
import subprocess
prof_dir = os.path.join(run_dir, 'density_profile')

def density_profile(system, nz=1000):
    # planar-averaged valence density (e/bohr^3) vs z (bohr, in [-c/2, c/2)); runs pp.x + average.x once
    out = os.path.join(prof_dir, 'avg_%s.dat' % system)
    if not os.path.isfile(out):
        os.makedirs(prof_dir, exist_ok=True)
        with open(os.path.join(prof_dir, 'pp_%s.in' % system), 'w') as f:
            f.write("&inputpp\n  prefix = 'gs_converged-%s'\n  outdir = '../'\n  filplot = 'rho_%s.dat'\n  plot_num = 0\n/\n"
                    % (system, system))
        with open(os.path.join(prof_dir, 'avg_%s.in' % system), 'w') as f:
            f.write('1\nrho_%s.dat\n1.0\n%d\n3\n1.0\n' % (system, nz))
        job = ['#!/bin/bash', '#SBATCH --job-name=density_profile', '#SBATCH --partition=all12h', '#SBATCH --nodes=1',
               '#SBATCH --ntasks-per-node=4', '#SBATCH --time=00:30:00', '#SBATCH --output=job_density_profile.out',
               'source %s' % module, 'mpirun -np 4 pp.x -inp pp_%s.in > pp_%s.log' % (system, system),
               'mpirun -np 1 average.x -inp avg_%s.in > avg_%s.log && mv avg.dat avg_%s.dat' % (system, system, system)]
        with open(os.path.join(prof_dir, 'job_density_profile.sh'), 'w') as f: f.write('\n'.join(job) + '\n')
        subprocess.run(['sbatch', '--wait', 'job_density_profile.sh'], cwd=prof_dir, check=True)
    z, rho, _ = np.loadtxt(out).T
    return np.where(z >= c / 2, z - c, z), rho

z, rho = density_profile('AA1A')
w = np.clip(rho, 0, None); w /= w.sum()
zc = np.angle((w * np.exp(2j * np.pi * z / c)).sum()) * c / (2 * np.pi)     # centre of the charge
zz = (z - zc + c / 2) % c - c / 2
m = rho > 1e-4 * rho.max()
print('electrons %.4f | rho > 1e-4 max over %.2f A (L/2 = %.2f A) | weight of the pairs at |z - z\'| > L/2: %.1e' % (
      rho.sum() * (z[1] - z[0]) * alat**2 * np.sqrt(3) / 2, np.ptp(zz[m]) * Const.Bohr_to_Angstrom,
      c / 2 * Const.Bohr_to_Angstrom, np.outer(w, w)[np.abs(zz[:, None] - zz[None, :]) > c / 2].sum()))

## NSCF on the 18x18x1 grid (for Yambo)

Bands: in `hBN_bilayer` the monolayer GW used 200 bands, which reach the same energy as about 250 bands in the
50 bohr cell (the high-energy states are vacuum states, their number grows with the cell volume); the bilayer and the
trilayer need about 15 and 30 more for the same window. The nscf runs keep a margin: 300 / 320 / 340 bands. The
exact numbers for Yambo are chosen in `GW_Analysis` (same energy window in the three systems). The three runs must
give the same 37 k vectors of the IBZ (overlaps between the systems and combination of the screening databases).

In [ ]:
nbnds = {'ML_top': 300, 'AA1': 320, 'AA1A': 340}
nscf_name = lambda s: 'kpoints_18-nbnds_%d-%s' % (nbnds[s], s)
study = D.Dataset(run_dir=run_dir, num_tasks=2)
study.set_postprocessing_function(PP.pw_parse_data)
for s in systems:
    inp = build_input(s, 18)
    inp.set_nscf(nbnds[s], force_symmorphic=True, conv_thr=1e-8)
    inp['electrons']['diagonalization'] = "'cg'"
    inp.set_prefix(nscf_name(s))
    study.append_run(id=s, input=inp, runner=qe_code(), name=nscf_name(s),
                     source_dir=os.path.join(run_dir, 'gs_converged-%s.save' % s))
res = study.run()
nscf = {s: res[i] for i, s in enumerate(systems)}

In [ ]:
kpts = {s: np.array(nscf[s].kpoints) for s in systems}
for s in systems:
    ev = np.array(nscf[s].get_evals(verbose=False))
    print('%-6s %d bands, %d k points in the IBZ | same k vectors as ML_top: %s | highest band %.1f-%.1f eV' % (
          s, ev.shape[1], len(ev), kpts[s].shape == kpts['ML_top'].shape and np.allclose(kpts[s], kpts['ML_top']),
          ev[:, -1].min(), ev[:, -1].max()))